# Tahap 5: Model Training

Notebook ini merepresentasikan blok abu-abu **Model Training** pada diagram metodologi penelitian, yang melatih 3 metode Topic Modeling menggunakan masing-masing representasi teks dari Tahap 4:
1. **LDA (*Latent Dirichlet Allocation*)** $\leftarrow$ Input: *Bag-of-Words*
2. **Top2Vec** $\leftarrow$ Input: *Semantic Vector Space*
3. **BERTopic** $\leftarrow$ Input: *Transformer Embedding*

In [ ]:
import os
import time
import pickle
import numpy as np
import pandas as pd
from gensim.models.ldamodel import LdaModel
from sklearn.metrics.pairwise import cosine_similarity
import umap
import hdbscan
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

os.makedirs('models', exist_ok=True)
NUM_TOPICS = 5  # Jumlah topik standar untuk perbandingan yang adil antar ketiga metode
TOP_N_WORDS = 10

## 5.0 Memuat Representasi Teks dari Tahap 4

In [ ]:
# 1. Muat Bag-of-Words
with open('representations/bow_representation.pkl', 'rb') as f:
    bow_data = pickle.load(f)

gensim_dict = bow_data['gensim_dict']
bow_corpus = bow_data['bow_corpus']
tokenized_docs = bow_data['tokenized_docs']
texts = bow_data['texts']

# 2. Muat Semantic Vector Space (untuk Top2Vec)
with open('representations/semantic_vector_space.pkl', 'rb') as f:
    semantic_data = pickle.load(f)

doc_vectors_semantic = semantic_data['doc_vectors']
word_vocab_semantic = semantic_data['word_vocab']
word_vectors_semantic = semantic_data['word_vectors']

# 3. Muat Transformer Embedding (untuk BERTopic)
transformer_embeddings = np.load('representations/transformer_embeddings.npy')

print('Seluruh representasi fitur berhasil dimuat!')

## 5.1 Model 1: LDA (Latent Dirichlet Allocation)
Melatih model probabilistik **LDA** menggunakan representasi *Bag-of-Words*.

In [ ]:
start_time = time.time()

lda_model = LdaModel(
    corpus=bow_corpus,
    id2word=gensim_dict,
    num_topics=NUM_TOPICS,
    random_state=42,
    passes=20,
    alpha='auto',
    eta='auto'
)

lda_time = time.time() - start_time

# Ekstrak Top-N kata dan bobot untuk tiap topik LDA
lda_topics = []
lda_topic_weights = []
for topic_id in range(NUM_TOPICS):
    words_weights = lda_model.show_topic(topic_id, topn=TOP_N_WORDS)
    lda_topics.append([w for w, _ in words_weights])
    lda_topic_weights.append(words_weights)

# Penugasan topik dominan per dokumen
lda_doc_topics = [
    max(lda_model.get_document_topics(bow), key=lambda x: x[1])[0] if bow else 0
    for bow in bow_corpus
]

print(f'Waktu Training LDA: {lda_time:.2f} detik\n')
for idx, words in enumerate(lda_topics):
    print(f'Topik {idx + 1} (LDA): {", ".join(words)}')

## 5.2 Model 2: Top2Vec
Melatih **Top2Vec** pada *Semantic Vector Space* (*Joint Word-Document Embedding*):
1. Reduksi dimensi vektor dokumen menggunakan **UMAP**.
2. Klasterisasi dokumen padat menggunakan **HDBSCAN**.
3. Reduksi hierarkis klaster menjadi $K$ topik utama dan ekstraksi kata kunci berdasarkan **Cosine Similarity** terdekat antara *Topic Centroid Vector* dengan *Word Vectors* di ruang semantik.

In [ ]:
start_time = time.time()

# 1. UMAP Dimensionality Reduction pada Document Vectors di Semantic Space
umap_top2vec = umap.UMAP(
    n_neighbors=15,
    n_components=5,
    metric='cosine',
    random_state=42
)
umap_doc_vecs = umap_top2vec.fit_transform(doc_vectors_semantic)

# 2. HDBSCAN Clustering untuk menemukan klaster padat
hdbscan_top2vec = hdbscan.HDBSCAN(
    min_cluster_size=15,
    metric='euclidean',
    cluster_selection_method='eom'
)
raw_labels = hdbscan_top2vec.fit_predict(umap_doc_vecs)

# 3. Hitung Topic Vectors (Centroid di ruang Semantic Vector Space) & Hierarchical Topic Reduction ke NUM_TOPICS
unique_clusters = [c for c in sorted(set(raw_labels)) if c != -1]
cluster_centroids = [
    doc_vectors_semantic[raw_labels == c].mean(axis=0) for c in unique_clusters
]
cluster_sizes = [int(np.sum(raw_labels == c)) for c in unique_clusters]

# Reduksi hierarkis (menggabungkan klaster terkecil ke klaster paling mirip hingga tersisa NUM_TOPICS)
while len(cluster_centroids) > NUM_TOPICS:
    smallest_idx = int(np.argmin(cluster_sizes))
    smallest_vec = cluster_centroids.pop(smallest_idx)
    smallest_size = cluster_sizes.pop(smallest_idx)
    
    sims = cosine_similarity([smallest_vec], cluster_centroids)[0]
    nearest_idx = int(np.argmax(sims))
    
    # Weighted average centroid
    total_size = cluster_sizes[nearest_idx] + smallest_size
    cluster_centroids[nearest_idx] = (
        cluster_centroids[nearest_idx] * cluster_sizes[nearest_idx] + smallest_vec * smallest_size
    ) / total_size
    cluster_sizes[nearest_idx] = total_size

topic_centroids_top2vec = np.array(cluster_centroids)

# 4. Cari Top-N Kata Kunci yang vektornya paling dekat dengan masing-masing Topic Centroid
sim_matrix = cosine_similarity(topic_centroids_top2vec, word_vectors_semantic)
top2vec_topics = []
top2vec_topic_weights = []

for topic_id in range(len(topic_centroids_top2vec)):
    top_indices = np.argsort(sim_matrix[topic_id])[::-1][:TOP_N_WORDS]
    words = [word_vocab_semantic[i] for i in top_indices]
    weights = [(word_vocab_semantic[i], float(sim_matrix[topic_id][i])) for i in top_indices]
    top2vec_topics.append(words)
    top2vec_topic_weights.append(weights)

# Penugasan topik dominan untuk setiap dokumen berdasarkan jarak ke Topic Centroid
doc_to_topic_sims = cosine_similarity(doc_vectors_semantic, topic_centroids_top2vec)
top2vec_doc_topics = np.argmax(doc_to_topic_sims, axis=1).tolist()

top2vec_time = time.time() - start_time
print(f'Waktu Training Top2Vec: {top2vec_time:.2f} detik\n')
for idx, words in enumerate(top2vec_topics):
    print(f'Topik {idx + 1} (Top2Vec): {", ".join(words)}')

## 5.3 Model 3: BERTopic
Melatih **BERTopic** menggunakan *Transformer Embedding* + reduksi dimensi **UMAP** + klasterisasi **HDBSCAN** + pembobotan kata per klaster dengan **c-TF-IDF (*Class-based TF-IDF*)**.

In [ ]:
start_time = time.time()

umap_bertopic = umap.UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

hdbscan_bertopic = hdbscan.HDBSCAN(
    min_cluster_size=15,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

vectorizer_bertopic = CountVectorizer(min_df=3)

bertopic_model = BERTopic(
    umap_model=umap_bertopic,
    hdbscan_model=hdbscan_bertopic,
    vectorizer_model=vectorizer_bertopic,
    nr_topics=NUM_TOPICS + 1,  # +1 karena BERTopic menyertakan topik outlier (-1) + NUM_TOPICS topik utama
    top_n_words=TOP_N_WORDS,
    calculate_probabilities=False,
    verbose=False
)

bertopic_doc_topics, _ = bertopic_model.fit_transform(texts, embeddings=transformer_embeddings)
bertopic_time = time.time() - start_time

# Ambil NUM_TOPICS topik utama (selain outlier -1)
valid_topic_ids = [t for t in bertopic_model.get_topic_info()['Topic'].tolist() if t != -1][:NUM_TOPICS]
bertopic_topics = []
bertopic_topic_weights = []

for t_id in valid_topic_ids:
    words_weights = bertopic_model.get_topic(t_id)[:TOP_N_WORDS]
    bertopic_topics.append([w for w, _ in words_weights])
    bertopic_topic_weights.append([(w, float(s)) for w, s in words_weights])

print(f'Waktu Training BERTopic: {bertopic_time:.2f} detik\n')
for idx, words in enumerate(bertopic_topics):
    print(f'Topik {idx + 1} (BERTopic): {", ".join(words)}')

## 5.4 Menyimpan Hasil Training Ketiga Model
Menyimpan daftar kata kunci topik, bobot kata, penugasan topik per dokumen, dan waktu komputasi ke `models/training_results.pkl` untuk dievaluasi di **`06_Model_Evaluation.ipynb`**.

In [ ]:
training_results = {
    'num_topics': NUM_TOPICS,
    'top_n_words': TOP_N_WORDS,
    'LDA': {
        'topics': lda_topics,
        'topic_weights': lda_topic_weights,
        'doc_topics': lda_doc_topics,
        'training_time': lda_time
    },
    'Top2Vec': {
        'topics': top2vec_topics,
        'topic_weights': top2vec_topic_weights,
        'doc_topics': top2vec_doc_topics,
        'training_time': top2vec_time
    },
    'BERTopic': {
        'topics': bertopic_topics,
        'topic_weights': bertopic_topic_weights,
        'doc_topics': bertopic_doc_topics,
        'training_time': bertopic_time
    }
}

with open('models/training_results.pkl', 'wb') as f:
    pickle.dump(training_results, f)

print('Seluruh hasil pelatihan model berhasil disimpan ke: models/training_results.pkl')